In [0]:
from pyspark.sql import functions as F

CATALOG = "healthcare_medallion_dbw"

appointments = spark.table(
    f"{CATALOG}.silver.appointments"
)

treatments = spark.table(
    f"{CATALOG}.silver.treatments"
)

billing = spark.table(
    f"{CATALOG}.silver.billing"
)

doctors = spark.table(
    f"{CATALOG}.silver.doctors"
)

In [0]:
appointment_kpis = (
    appointments
    .agg(
        F.count("*").alias("total_appointments"),

        F.sum("no_show_flag").alias("no_show_count"),

        F.sum("completed_flag").alias("completed_count"),

        F.sum("cancelled_flag").alias("cancelled_count")
    )
    .withColumn(
        "no_show_rate",
        F.round(
            F.col("no_show_count")
            * 100.0
            / F.col("total_appointments"),
            2
        )
    )
    .withColumn(
        "completion_rate",
        F.round(
            F.col("completed_count")
            * 100.0
            / F.col("total_appointments"),
            2
        )
    )
    .withColumn(
        "cancellation_rate",
        F.round(
            F.col("cancelled_count")
            * 100.0
            / F.col("total_appointments"),
            2
        )
    )
)

display(appointment_kpis)

total_appointments,no_show_count,completed_count,cancelled_count,no_show_rate,completion_rate,cancellation_rate
200,52,46,51,26.0,23.0,25.5


In [0]:
treatment_kpis = (
    treatments
    .agg(
        F.count("*").alias("total_treatments"),

        F.round(
            F.sum("cost"),
            2
        ).alias("total_treatment_cost"),

        F.round(
            F.avg("cost"),
            2
        ).alias("average_treatment_cost")
    )
)

display(treatment_kpis)

total_treatments,total_treatment_cost,average_treatment_cost
200,551249.85,2756.25


In [0]:
billing_kpis = (
    billing
    .agg(
        F.count("*").alias("total_bills"),

        F.round(
            F.sum("amount"),
            2
        ).alias("total_billing_amount"),

        F.sum(
            "payment_success_flag"
        ).alias("paid_bill_count"),

        F.sum(
            "pending_flag"
        ).alias("pending_bill_count"),

        F.sum(
            "failed_payment_flag"
        ).alias("failed_bill_count")
    )

    .withColumn(
        "payment_success_rate",
        F.round(
            F.col("paid_bill_count")
            * 100.0
            / F.col("total_bills"),
            2
        )
    )

    .withColumn(
        "failed_payment_rate",
        F.round(
            F.col("failed_bill_count")
            * 100.0
            / F.col("total_bills"),
            2
        )
    )
)

display(billing_kpis)

total_bills,total_billing_amount,paid_bill_count,pending_bill_count,failed_bill_count,payment_success_rate,failed_payment_rate
200,551249.85,64,69,67,32.0,33.5


In [0]:
pending_amount = (
    billing
    .filter(
        F.col("payment_status") == "PENDING"
    )
    .agg(
        F.round(
            F.sum("amount"),
            2
        ).alias("pending_billing_amount")
    )
)

display(pending_amount)

pending_billing_amount
184612.01


In [0]:
payment_method_revenue = (
    billing

    .groupBy(
        "payment_method"
    )

    .agg(
        F.count("*").alias(
            "bill_count"
        ),

        F.round(
            F.sum("amount"),
            2
        ).alias(
            "total_revenue"
        )
    )

    .orderBy(
        F.col(
            "total_revenue"
        ).desc()
    )
)

display(payment_method_revenue)

payment_method,bill_count,total_revenue
CREDIT CARD,75,201382.43
INSURANCE,64,182160.28
CASH,61,167707.14


In [0]:
treatment_type_kpis = (
    treatments

    .groupBy(
        "treatment_type"
    )

    .agg(
        F.count("*").alias(
            "treatment_count"
        ),

        F.round(
            F.avg("cost"),
            2
        ).alias(
            "average_cost"
        ),

        F.round(
            F.sum("cost"),
            2
        ).alias(
            "total_cost"
        )
    )

    .orderBy(
        F.col(
            "total_cost"
        ).desc()
    )
)

display(treatment_type_kpis)

treatment_type,treatment_count,average_cost,total_cost
Chemotherapy,49,2629.71,128855.68
Mri,36,3224.95,116098.16
X-ray,41,2698.87,110653.67
Physiotherapy,36,2761.61,99418.1
Ecg,38,2532.22,96224.24


In [0]:
doctor_appointments = (
    appointments

    .join(
        doctors.select(
            "doctor_id",
            "first_name",
            "last_name",
            "specialization",
            "hospital_branch"
        ),
        on="doctor_id",
        how="left"
    )

    .groupBy(
        "doctor_id",
        "first_name",
        "last_name",
        "specialization",
        "hospital_branch"
    )

    .agg(
        F.count(
            "appointment_id"
        ).alias(
            "appointment_count"
        ),

        F.sum(
            "no_show_flag"
        ).alias(
            "no_show_count"
        )
    )

    .withColumn(
        "doctor_no_show_rate",
        F.round(
            F.col("no_show_count")
            * 100.0
            / F.col("appointment_count"),
            2
        )
    )

    .orderBy(
        F.col(
            "appointment_count"
        ).desc()
    )
)

display(doctor_appointments)

doctor_id,first_name,last_name,specialization,hospital_branch,appointment_count,no_show_count,doctor_no_show_rate
D005,Sarah,Taylor,Dermatology,Central Hospital,29,9,31.03
D001,David,Taylor,Dermatology,Westside Clinic,25,7,28.0
D006,Alex,Davis,Pediatrics,Central Hospital,24,6,25.0
D003,Jane,Smith,Pediatrics,Eastside Clinic,22,7,31.82
D002,Jane,Davis,Pediatrics,Eastside Clinic,21,1,4.76
D010,Linda,Wilson,Oncology,Eastside Clinic,19,5,26.32
D009,Sarah,Smith,Pediatrics,Central Hospital,17,6,35.29
D008,Linda,Brown,Dermatology,Westside Clinic,16,4,25.0
D004,David,Jones,Pediatrics,Central Hospital,14,5,35.71
D007,Robert,Davis,Oncology,Westside Clinic,13,2,15.38


In [0]:
summary_rows = []

appointment_row = appointment_kpis.first()
treatment_row = treatment_kpis.first()
billing_row = billing_kpis.first()
pending_row = pending_amount.first()

summary_rows.extend([
    (
        "Patient No-Show Rate",
        float(
            appointment_row[
                "no_show_rate"
            ]
        ),
        "PERCENT"
    ),

    (
        "Appointment Completion Rate",
        float(
            appointment_row[
                "completion_rate"
            ]
        ),
        "PERCENT"
    ),

    (
        "Appointment Cancellation Rate",
        float(
            appointment_row[
                "cancellation_rate"
            ]
        ),
        "PERCENT"
    ),

    (
        "Total Treatment Cost",
        float(
            treatment_row[
                "total_treatment_cost"
            ]
        ),
        "CURRENCY"
    ),

    (
        "Average Treatment Cost",
        float(
            treatment_row[
                "average_treatment_cost"
            ]
        ),
        "CURRENCY"
    ),

    (
        "Payment Success Rate",
        float(
            billing_row[
                "payment_success_rate"
            ]
        ),
        "PERCENT"
    ),

    (
        "Failed Payment Rate",
        float(
            billing_row[
                "failed_payment_rate"
            ]
        ),
        "PERCENT"
    ),

    (
        "Pending Billing Amount",
        float(
            pending_row[
                "pending_billing_amount"
            ]
            or 0
        ),
        "CURRENCY"
    )
])

In [0]:
gold_kpi_summary = (
    spark.createDataFrame(
        summary_rows,
        [
            "kpi_name",
            "kpi_value",
            "kpi_unit"
        ]
    )

    .withColumn(
        "_gold_load_timestamp",
        F.current_timestamp()
    )

    .withColumn(
        "_gold_batch_id",
        F.expr("uuid()")
    )

    .withColumn(
        "_kpi_period",
        F.lit("DAILY")
    )

    .withColumn(
        "_kpi_version",
        F.lit(1)
    )

    .withColumn(
        "_report_as_of_date",
        F.current_date()
    )

    .withColumn(
        "_is_restatement",
        F.lit(False)
    )
)

display(gold_kpi_summary)

kpi_name,kpi_value,kpi_unit,_gold_load_timestamp,_gold_batch_id,_kpi_period,_kpi_version,_report_as_of_date,_is_restatement
Patient No-Show Rate,26.0,PERCENT,2026-08-10T17:32:40.708Z,e9e41e98-0beb-4804-bcc0-e3681f2d9c6f,DAILY,1,2026-08-10,false
Appointment Completion Rate,23.0,PERCENT,2026-08-10T17:32:40.708Z,e0ff660a-183d-4278-b373-49a736ed774a,DAILY,1,2026-08-10,false
Appointment Cancellation Rate,25.5,PERCENT,2026-08-10T17:32:40.708Z,037cd6a9-69ba-4d8b-a61f-a9d8cac4f651,DAILY,1,2026-08-10,false
Total Treatment Cost,551249.85,CURRENCY,2026-08-10T17:32:40.708Z,7aecceed-fbc8-49f9-b3a3-88f2c57c5d63,DAILY,1,2026-08-10,false
Average Treatment Cost,2756.25,CURRENCY,2026-08-10T17:32:40.708Z,52c661d9-5fef-4711-83ca-ad6bc464848c,DAILY,1,2026-08-10,false
Payment Success Rate,32.0,PERCENT,2026-08-10T17:32:40.708Z,93859d86-33b5-42ea-9a45-2ab01fa12c0d,DAILY,1,2026-08-10,false
Failed Payment Rate,33.5,PERCENT,2026-08-10T17:32:40.708Z,afa7a9ed-40a6-4a3a-8b69-b1f9419e42dc,DAILY,1,2026-08-10,false
Pending Billing Amount,184612.01,CURRENCY,2026-08-10T17:32:40.708Z,64d215ca-a3da-40fa-8ef6-3d8b16f0df58,DAILY,1,2026-08-10,false


In [0]:
(
    gold_kpi_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG}.gold.kpi_summary"
    )
)

In [0]:
(
    payment_method_revenue.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG}.gold.payment_method_revenue"
    )
)

In [0]:
(
    treatment_type_kpis.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG}.gold.treatment_type_kpis"
    )
)

In [0]:
(
    doctor_appointments.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG}.gold.doctor_appointments"
    )
)

In [0]:
spark.sql(
    f"SHOW TABLES IN {CATALOG}.gold"
).show(truncate=False)

+--------+----------------------+-----------+
|database|tableName             |isTemporary|
+--------+----------------------+-----------+
|gold    |doctor_appointments   |false      |
|gold    |kpi_summary           |false      |
|gold    |payment_method_revenue|false      |
|gold    |treatment_type_kpis   |false      |
+--------+----------------------+-----------+



In [0]:
display(
    spark.table(
        f"{CATALOG}.gold.kpi_summary"
    )
)

kpi_name,kpi_value,kpi_unit,_gold_load_timestamp,_gold_batch_id,_kpi_period,_kpi_version,_report_as_of_date,_is_restatement
Patient No-Show Rate,26.0,PERCENT,2026-08-10T17:32:52.380Z,e9e41e98-0beb-4804-bcc0-e3681f2d9c6f,DAILY,1,2026-08-10,false
Appointment Completion Rate,23.0,PERCENT,2026-08-10T17:32:52.380Z,e0ff660a-183d-4278-b373-49a736ed774a,DAILY,1,2026-08-10,false
Appointment Cancellation Rate,25.5,PERCENT,2026-08-10T17:32:52.380Z,037cd6a9-69ba-4d8b-a61f-a9d8cac4f651,DAILY,1,2026-08-10,false
Total Treatment Cost,551249.85,CURRENCY,2026-08-10T17:32:52.380Z,7aecceed-fbc8-49f9-b3a3-88f2c57c5d63,DAILY,1,2026-08-10,false
Average Treatment Cost,2756.25,CURRENCY,2026-08-10T17:32:52.380Z,52c661d9-5fef-4711-83ca-ad6bc464848c,DAILY,1,2026-08-10,false
Payment Success Rate,32.0,PERCENT,2026-08-10T17:32:52.380Z,93859d86-33b5-42ea-9a45-2ab01fa12c0d,DAILY,1,2026-08-10,false
Failed Payment Rate,33.5,PERCENT,2026-08-10T17:32:52.380Z,afa7a9ed-40a6-4a3a-8b69-b1f9419e42dc,DAILY,1,2026-08-10,false
Pending Billing Amount,184612.01,CURRENCY,2026-08-10T17:32:52.380Z,64d215ca-a3da-40fa-8ef6-3d8b16f0df58,DAILY,1,2026-08-10,false
